# Decision Tree — PaySim Fraud Detection (Official Pipeline)

Pipeline chính thức DCAI: **Step 2** (LOO ablation tự động chọn từ 10 PIT-safe features) → **Step 3** (walk-forward CV) → **SHAP** (giải thích model) → **Step 5b** (baseline + Fβ(1.75) threshold tuning) → **MLflow** (register model).

> ℹ️ `BETA = 1.75` là tham số chính thức xuyên suốt pipeline — sweep threshold theo Fβ(1.75) trên validation set.


In [13]:
# === SETUP: Shared config & data loading ===
import numpy as np
import pandas as pd
import gc, time
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt

RANDOM_STATE = 42
BETA = 1.75  # Fβ optimization target cho threshold tuning (xuyên suốt pipeline)

# Load data
DATA = "data/transactions_train.csv"
dtypes = {
    'step': 'int32', 'type': 'category', 'amount': 'float32',
    'nameOrig': 'category', 'oldbalanceOrg': 'float32', 'newbalanceOrig': 'float32',
    'nameDest': 'category', 'oldbalanceDest': 'float32', 'newbalanceDest': 'float32',
    'isFraud': 'int8', 'isFlaggedFraud': 'int8'
}

# Model hyperparameters (fixed across all experiments)
DT_PARAMS = dict(
    max_depth=6,
    min_samples_leaf=50,
    min_samples_split=100,
    class_weight='balanced',
    random_state=RANDOM_STATE
)

print(f"✓ Setup complete. DT_PARAMS={DT_PARAMS} | BETA={BETA}")

✓ Setup complete. DT_PARAMS={'max_depth': 6, 'min_samples_leaf': 50, 'min_samples_split': 100, 'class_weight': 'balanced', 'random_state': 42} | BETA=1.75


### 10 PIT-safe Features

| # | Feature | Type | Ý nghĩa |
|---|---------|------|---------|
| 1 | `step_day` | Derived | Ngày mô phỏng (step // 24) |
| 2 | `hour_day` | Derived | Giờ trong ngày (step % 24) |
| 3 | `type_code` | Encoded | Loại giao dịch (categorical → int) |
| 4 | `is_customer_dest` | Derived | Người nhận là khách hàng cá nhân (C=1, M=0) |
| 5 | `amount_log` | Transformed | log1p(amount) — nén skew |
| 6 | `amount_ratio` | Derived | amount / median(amount theo type) — chuẩn hoá theo loại GD |
| 7 | `dest_freq` | Aggregate | Tần suất xuất hiện của nameDest trên train |
| 8 | `dest_amount_mean` | Aggregate | Trung bình amount của nameDest trên train |
| 9 | `dest_type_count` | Aggregate | Số loại GD khác nhau của nameDest trên train |
| 10 | `dest_cashout_freq` | Aggregate | Số lần CASH_OUT của nameDest trên train |

> ⚠️ **Không dùng balance columns** (`oldbalanceOrg`, `newbalanceOrig`, `oldbalanceDest`, `newbalanceDest`) — EDA §8 chứng minh đây là **leakage**: post-event values tạo fingerprint bắt 100% fraud, model không generalize được.


### Pipeline chọn features tự động 

Pipeline tự động chọn:

1. **Step 2**: Build 10 PIT-safe features, LOO ablation trên random-split → tự động giữ features có ΔPR-AUC ≥ 0 (chỉ bỏ delta âm) → gán vào `FEATURES_CANDIDATES`.
2. **Step 3**: Walk-forward CV trên `FEATURES_CANDIDATES` → đo PR-AUC temporal stability.
3. **Step 4+**: SHAP, final model đều dùng `FEATURES_CANDIDATES` từ Step 2.


In [17]:
# === BƯỚC 2 (DCAI): Feature Engineering + LOO Ablation tự động chọn features ===
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score

# --- reload df ---
df = pd.read_csv(DATA, dtype=dtypes)
df["step_day"] = (df["step"] // 24).astype("int16")
df["hour_day"] = (df["step"] % 24).astype("int8")
df["type_code"] = df["type"].astype("category").cat.codes.astype("int8")
df["is_customer_dest"] = df["nameDest"].str.startswith("C").astype("int8")

y2 = df["isFraud"].to_numpy()
X2 = df[["step_day", "hour_day", "type_code", "is_customer_dest", "amount", "type", "nameDest"]].copy()

# --- split TRƯỚC khi engineering (PIT-safe) ---
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=RANDOM_STATE, stratify=y2
)

def build_step2_features(Xtr, Xte):
    """Build all 10 PIT-safe candidate features."""
    for Xp in (Xtr, Xte):
        Xp["amount_log"] = np.log1p(Xp["amount"]).astype("float32")
    med2 = Xtr.groupby("type", observed=True)["amount"].median().reindex(Xtr["type"].cat.categories).to_numpy()
    for Xp in (Xtr, Xte):
        Xp["amount_ratio"] = (Xp["amount"].to_numpy() / med2[Xp["type"].cat.codes.to_numpy()]).astype("float32")
    d_freq2 = Xtr["nameDest"].value_counts()
    d_mean2 = Xtr.groupby("nameDest", observed=True)["amount"].mean()
    d_types2 = Xtr.groupby("nameDest", observed=True)["type"].nunique()
    d_co2 = Xtr.loc[Xtr["type"] == "CASH_OUT"].groupby("nameDest", observed=True).size()
    for Xp in (Xtr, Xte):
        Xp["dest_freq"] = Xp["nameDest"].map(d_freq2).fillna(0).astype("int32").to_numpy()
        Xp["dest_amount_mean"] = Xp["nameDest"].map(d_mean2).fillna(0).astype("float32").to_numpy()
        Xp["dest_type_count"] = Xp["nameDest"].map(d_types2).fillna(0).astype("int8").to_numpy()
        Xp["dest_cashout_freq"] = Xp["nameDest"].map(d_co2).fillna(0).astype("int32").to_numpy()
    return Xtr, Xte

X2_train, X2_test = build_step2_features(X2_train, X2_test)

# === 10 PIT-SAFE FEATURES CANDIDATES ===
ALL_FEATURES_10 = [
    "step_day", "hour_day", "type_code", "is_customer_dest",
    "amount_log", "amount_ratio",
    "dest_freq", "dest_amount_mean", "dest_type_count", "dest_cashout_freq"
]

print(f"★ 10 PIT-safe features candidates: {ALL_FEATURES_10}")

del df, X2
gc.collect()
print(f"X2_train: {X2_train.shape} | X2_test: {X2_test.shape}")

# --- eval function ---
def eval_test(features, label, Xtr, ytr, Xte, yte):
    """Fit + evaluate với PR-AUC, Recall@1%, Precision@1%."""
    m = DecisionTreeClassifier(**DT_PARAMS)
    m.fit(Xtr[features], ytr)
    prob = m.predict_proba(Xte[features])[:, 1]
    pr = average_precision_score(yte, prob)
    n_top1 = max(1, int(np.ceil(len(yte) * 0.01)))
    top_idx = np.argsort(prob)[::-1][:n_top1]
    recall1 = yte[top_idx].sum() / yte.sum()
    precision1 = yte[top_idx].mean()
    print(f"{label:35s} | {len(features):2d} feat | PR-AUC {pr:.4f} | R@1% {recall1*100:.2f}% | P@1% {precision1*100:.2f}%")
    return m, pr

# === LOO ABLATION trên 10 features (random-split) ===
print()
print("=== LOO Ablation (drop 1 feature at a time) ===")
_, pr_all = eval_test(ALL_FEATURES_10, "ALL 10 features (baseline)", X2_train, y2_train, X2_test, y2_test)

loo_results = {}
for feat in ALL_FEATURES_10:
    subset = [f for f in ALL_FEATURES_10 if f != feat]
    _, pr_loo = eval_test(subset, f"Drop {feat}", X2_train, y2_train, X2_test, y2_test)
    loo_results[feat] = pr_loo

# Chọn features: chỉ bỏ những feature delta ÂM (có hại); giữ lại delta >= 0
print()
print("=== LOO Delta (PR-AUC change when dropped) ===")
deltas = {feat: pr_all - pr_loo for feat, pr_loo in loo_results.items()}
for feat, delta in sorted(deltas.items(), key=lambda x: -x[1]):
    status = "KEEP" if delta >= 0 else "DROP"
    print(f"  {feat:25s}: delta = {delta:+.4f} -> {status}")

# Tự động chọn: giữ features có positive delta
FEATURES_SELECTED = [f for f in ALL_FEATURES_10 if deltas[f] >= 0]
print()
print(f"★ Auto-selected ({len(FEATURES_SELECTED)} features): {FEATURES_SELECTED}")

# Eval bộ selected
_, pr_selected = eval_test(FEATURES_SELECTED, f"Selected {len(FEATURES_SELECTED)} features", X2_train, y2_train, X2_test, y2_test)

# Feature importance trên bộ selected
m_sel = DecisionTreeClassifier(**DT_PARAMS)
m_sel.fit(X2_train[FEATURES_SELECTED], y2_train)
print()
print("Feature importance (selected):")
imp = pd.Series(m_sel.feature_importances_, index=FEATURES_SELECTED).sort_values(ascending=False)
print(imp.round(4).to_string())

# Lưu cho các bước sau (SHAP, Final Model)
FEATURES_CANDIDATES = FEATURES_SELECTED.copy()
print()
print(f"-> FEATURES_CANDIDATES ({len(FEATURES_CANDIDATES)}) chuyển sang Step 3 walk-forward CV")


★ 10 PIT-safe features candidates: ['step_day', 'hour_day', 'type_code', 'is_customer_dest', 'amount_log', 'amount_ratio', 'dest_freq', 'dest_amount_mean', 'dest_type_count', 'dest_cashout_freq']
X2_train: (5090096, 13) | X2_test: (1272524, 13)

=== LOO Ablation (drop 1 feature at a time) ===
ALL 10 features (baseline)          | 10 feat | PR-AUC 0.1661 | R@1% 58.67% | P@1% 7.58%
Drop step_day                       |  9 feat | PR-AUC 0.1719 | R@1% 59.28% | P@1% 7.65%
Drop hour_day                       |  9 feat | PR-AUC 0.1611 | R@1% 55.26% | P@1% 7.13%
Drop type_code                      |  9 feat | PR-AUC 0.1188 | R@1% 44.49% | P@1% 5.74%
Drop is_customer_dest               |  9 feat | PR-AUC 0.1661 | R@1% 58.67% | P@1% 7.58%
Drop amount_log                     |  9 feat | PR-AUC 0.2271 | R@1% 68.78% | P@1% 8.88%
Drop amount_ratio                   |  9 feat | PR-AUC 0.1718 | R@1% 54.53% | P@1% 7.04%
Drop dest_freq                      |  9 feat | PR-AUC 0.1500 | R@1% 49.54% | P@1% 

## Step 3 — Walk-forward CV (đo ổn định theo thời gian)

Bước ③ của pipeline: thay random split bằng **expanding window theo `step`** — fold k train trên các bước thời gian đầu, test trên đoạn liền sau (không nhìn tương lai). Mục tiêu:

1. **Đo ổn định theo thời gian**: PR-AUC của bộ `FEATURES_CANDIDATES` (từ Step 2 LOO ablation) có giữ được trên các "tương lai" liên tiếp không, hay chỉ đẹp vì random split nhìn thấy đủ mọi thời điểm.
2. **So sánh trên từng fold**: PR-AUC thắng đều ở mọi fold mới là ổn định thật, không phải may mắn trên 1 test set.

PIT-safe như bước 2: mọi aggregate (median theo type, thống kê `dest_*`) tính từ TRAIN của **từng fold** rồi map sang test fold đó. Mỗi fold = train expanding từ đầu timeline, test là 1/6 dải thời gian liền sau. ⏱ cell này chạy ~5–7 phút.


In [21]:
# === BƯỚC 3 (DCAI): Walk-forward CV — đo ổn định theo thời gian ===
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score

# --- reload df (bước 2 đã del df) ---
df = pd.read_csv(DATA, dtype=dtypes)
df["step_day"] = (df["step"] // 24).astype("int16")
df["hour_day"] = (df["step"] % 24).astype("int8")
df["type_code"] = df["type"].astype("category").cat.codes.astype("int8")
base = df[["step", "step_day", "hour_day", "type_code", "amount", "type", "nameDest", "isFraud"]].copy()
del df
gc.collect()

print(f"FEATURES_CANDIDATES ({len(FEATURES_CANDIDATES)} candidates từ Step 2): {FEATURES_CANDIDATES}")

def build_features(tr, te):
    """PIT-safe: aggregate từ TRAIN của fold này, map sang test fold."""
    for Xp in (tr, te):
        Xp["is_customer_dest"] = Xp["nameDest"].str.startswith("C").astype("int8")
        Xp["amount_log"] = np.log1p(Xp["amount"]).astype("float32")
    med = tr.groupby("type", observed=True)["amount"].median().reindex(tr["type"].cat.categories).to_numpy()
    for Xp in (tr, te):
        Xp["amount_ratio"] = (Xp["amount"].to_numpy() / med[Xp["type"].cat.codes.to_numpy()]).astype("float32")
    d_freq = tr["nameDest"].value_counts()
    d_mean = tr.groupby("nameDest", observed=True)["amount"].mean()
    d_types = tr.groupby("nameDest", observed=True)["type"].nunique()
    d_co = tr.loc[tr["type"] == "CASH_OUT"].groupby("nameDest", observed=True).size()
    for Xp in (tr, te):
        Xp["dest_freq"] = Xp["nameDest"].map(d_freq).fillna(0).astype("int32").to_numpy()
        Xp["dest_amount_mean"] = Xp["nameDest"].map(d_mean).fillna(0).astype("float32").to_numpy()
        Xp["dest_type_count"] = Xp["nameDest"].map(d_types).fillna(0).astype("int8").to_numpy()
        Xp["dest_cashout_freq"] = Xp["nameDest"].map(d_co).fillna(0).astype("int32").to_numpy()
    return tr, te

def eval_pr(feat, Xtr, ytr, Xte, yte):
    m = DecisionTreeClassifier(**DT_PARAMS)
    m.fit(Xtr[feat], ytr)
    p = m.predict_proba(Xte[feat])[:, 1]
    pr = average_precision_score(yte, p)
    return pr, m

N_FOLDS = 5
step_min, step_max = int(base["step"].min()), int(base["step"].max())
segs = np.linspace(step_min, step_max + 1, N_FOLDS + 2).astype(int)
print(f"Walk-forward expanding, {N_FOLDS} folds | steps [{step_min}, {step_max}] | segs: {segs.tolist()}")

rows = []
for k in range(N_FOLDS):
    t1, t2 = segs[k + 1], segs[k + 2]
    tr = base[(base["step"] >= segs[0]) & (base["step"] < t1)].copy()
    te = base[(base["step"] >= t1) & (base["step"] < t2)].copy()
    tr, te = build_features(tr, te)
    ytr, yte = tr["isFraud"].to_numpy(), te["isFraud"].to_numpy()
    pr_chot, m = eval_pr(FEATURES_CANDIDATES, tr, ytr, te, yte)
    n_top1 = max(1, int(np.ceil(len(yte) * 0.01)))
    top = np.argsort(m.predict_proba(te[FEATURES_CANDIDATES])[:, 1])[::-1][:n_top1]
    recall1 = yte[top].sum() / yte.sum()
    rows.append({"fold": k + 1, "test_steps": f"{t1}-{t2 - 1}",
                 "n_train": len(tr), "n_test": len(te), "fraud_rate_test": yte.mean(),
                 "PR_AUC": pr_chot, "Recall@1%": recall1})
    msg = f"fold {k+1}: train n={len(tr):,} | test {t1}-{t2-1} n={len(te):,} "
    msg += f"fraud%={yte.mean()*100:.3f} | PR-AUC={pr_chot:.4f} | R@1% {recall1*100:.2f}"
    print(msg)
    del tr, te
    gc.collect()

wf = pd.DataFrame(rows)
print()
print("=== TỔNG HỢP walk-forward (5 folds, expanding) ===")
print(wf.round(4).to_string(index=False))
mean_c, std_c = wf["PR_AUC"].mean(), wf["PR_AUC"].std()
print(f"PR-AUC mean+/-std: {mean_c:.4f}+/-{std_c:.4f}")
print(f"★ KẾT LUẬN: walk-forward PR-AUC={mean_c:.4f}+/-{std_c:.4f}")


FEATURES_CANDIDATES (6 candidates từ Step 2): ['hour_day', 'type_code', 'is_customer_dest', 'dest_freq', 'dest_amount_mean', 'dest_type_count']
Walk-forward expanding, 5 folds | steps [1, 743] | segs: [1, 124, 248, 372, 496, 620, 744]
fold 1: train n=1,070,132 | test 124-247 n=2,123,087 fraud%=0.063 | PR-AUC=0.0071 | R@1% 16.08
fold 2: train n=3,193,219 | test 248-371 n=2,063,154 fraud%=0.066 | PR-AUC=0.1446 | R@1% 43.84
fold 3: train n=5,256,373 | test 372-495 n=798,411 fraud%=0.171 | PR-AUC=0.1654 | R@1% 50.66
fold 4: train n=6,054,784 | test 496-619 n=216,999 fraud%=0.613 | PR-AUC=0.2191 | R@1% 39.25
fold 5: train n=6,271,783 | test 620-743 n=90,837 fraud%=1.513 | PR-AUC=0.2615 | R@1% 23.22

=== TỔNG HỢP walk-forward (5 folds, expanding) ===
 fold test_steps  n_train  n_test  fraud_rate_test  PR_AUC  Recall@1%
    1    124-247  1070132 2123087           0.0006  0.0071     0.1608
    2    248-371  3193219 2063154           0.0007  0.1446     0.4384
    3    372-495  5256373  798411  

## Step 4 — SHAP giải thích model chốt

Bước ④ của pipeline: **giải thích model chốt** bằng SHAP (TreeExplainer — chính xác cho cây quyết định). Mục tiêu: hiểu từng feature đóng góp gì vào quyết định fraud.

- Model được giải thích: bộ `FEATURES_CANDIDATES` từ **Step 2** fit trên random-split train.
- Hai góc nhìn: **(1) global** — mean|SHAP| trên 3000 dòng test đại diện; **(2) trên fraud thật** — với các ca model cần nhận diện.
- **feature_importance ≠ SHAP**: gain-based (chia sâu = dễ over-rate); SHAP đo đóng góp thực trên từng dự đoán.
- Không gọi MLflow (server chung).


In [22]:
# === BƯỚC 4 (DCAI): SHAP giải thích model chốt — không MLflow ===
%pip install -q shap
import shap

# tái sử dụng X2_train/X2_test từ bước 2; cắt về bộ chốt FEATURES_CANDIDATES
Xtr5 = X2_train[FEATURES_CANDIDATES].copy()
Xte5 = X2_test[FEATURES_CANDIDATES].copy()
m5 = DecisionTreeClassifier(**DT_PARAMS)
m5.fit(Xtr5, y2_train)
pr5 = average_precision_score(y2_test, m5.predict_proba(Xte5)[:, 1])
print(f"chốt PR-AUC test (tái tạo, khớp bước 2): {pr5:.4f}")
print(f"FEATURES_CANDIDATES: {FEATURES_CANDIDATES} ({len(FEATURES_CANDIDATES)} features)")

# SHAP trên (1) mẫu đại diện 3000 dòng test + (2) TOÀN bộ fraud thật của test
rng = np.random.default_rng(RANDOM_STATE)
idx_rep = rng.choice(len(y2_test), size=3000, replace=False)
idx_fraud = np.where(y2_test == 1)[0]
X_rep, X_fraud = Xte5.iloc[idx_rep].copy(), Xte5.iloc[idx_fraud].copy()
print(f"SHAP mẫu đại diện n={len(X_rep)} (fraud={y2_test[idx_rep].sum()}) | fraud thật toàn bộ n={len(X_fraud)}")

explainer = shap.TreeExplainer(m5)
sv_rep = np.asarray(explainer.shap_values(X_rep))
sv_fraud = np.asarray(explainer.shap_values(X_fraud))
if sv_rep.ndim == 3:  # sklearn classifier -> [class0, class1], lấy class1 (fraud)
    sv_rep, sv_fraud = sv_rep[:, :, 1], sv_fraud[:, :, 1]

# --- 1) Global: mean|SHAP| vs feature_importance ---
g5 = pd.Series(np.abs(sv_rep).mean(axis=0), index=FEATURES_CANDIDATES).sort_values(ascending=False)
print("\n=== GLOBAL — mean|SHAP| (3000 dòng đại diện) vs feature_importance ===")
for f, v in g5.items():
    fi = m5.feature_importances_[FEATURES_CANDIDATES.index(f)]
    print(f"  {f:20s} mean|SHAP| {v:.5f}  (feature_importance {fi:.4f})")

# --- 2) Trên fraud thật: feature nào đẩy model 'tin fraud' ---
f5 = pd.Series(sv_fraud.mean(axis=0), index=FEATURES_CANDIDATES).sort_values(ascending=False)
print("\n=== TRÊN FRAUD THẬT — SHAP trung bình (đẩy lên = hỗ trợ nhận diện) ===")
for f, v in f5.items():
    print(f"  {f:20s} {v:+.5f}")

# --- 3) hour_day analysis (nếu có trong FEATURES_CANDIDATES) ---
if "hour_day" in FEATURES_CANDIDATES:
    i_h = FEATURES_CANDIDATES.index("hour_day")
    prof = pd.DataFrame({"hour": X_rep["hour_day"].to_numpy(), "shap": sv_rep[:, i_h]})
    prof_g = prof.groupby("hour")["shap"].mean().sort_values(ascending=False)
    print("\n=== hour_day — đóng góp theo giờ (mẫu đại diện) ===")
    print(prof_g.round(4).to_string())
    neg_hr = set(prof_g[prof_g < 0].index)
    frh = X_fraud["hour_day"].to_numpy(); svfh = sv_fraud[:, i_h]
    in_neg = np.isin(frh, list(neg_hr))
    print(f"\n★ hour_day: giờ SHAP>0 (model tin fraud): {list(prof_g[prof_g > 0].index)}")
    print(f"  giờ SHAP<0 (model nghi ngờ): {sorted(neg_hr)}")
    if in_neg.any():
        print(f"  {in_neg.mean()*100:.1f}% fraud THẬT nằm ở giờ bị hour_day đẩy XUỐNG " \
              f"(SHAP TB {svfh[in_neg].mean():+.4f} | giờ còn lại {svfh[~in_neg].mean():+.4f})")
    else:
        print(f"  Không có fraud thật nằm ở giờ SHAP<0")
else:
    print("\n★ hour_day KHÔNG nằm trong FEATURES_CANDIDATES → bỏ qua phân tích hour_day")

# chốt ý
print(f"\n★ KẾT LUẬN SHAP: bộ chốt {len(FEATURES_CANDIDATES)} features")
for f, v in g5.items():
    fi = m5.feature_importances_[FEATURES_CANDIDATES.index(f)]
    rank = list(g5.index).index(f) + 1
    print(f"  {rank}. {f:20s} mean|SHAP| {v:.4f} (importance {fi:.4f})")


Note: you may need to restart the kernel to use updated packages.
chốt PR-AUC test (tái tạo, khớp bước 2): 0.1632
FEATURES_CANDIDATES: ['hour_day', 'type_code', 'is_customer_dest', 'dest_freq', 'dest_amount_mean', 'dest_type_count'] (6 features)
SHAP mẫu đại diện n=3000 (fraud=2) | fraud thật toàn bộ n=1643

=== GLOBAL — mean|SHAP| (3000 dòng đại diện) vs feature_importance ===
  type_code            mean|SHAP| 0.28312  (feature_importance 0.5346)
  hour_day             mean|SHAP| 0.07018  (feature_importance 0.2679)
  dest_amount_mean     mean|SHAP| 0.05969  (feature_importance 0.1118)
  dest_freq            mean|SHAP| 0.04543  (feature_importance 0.0773)
  dest_type_count      mean|SHAP| 0.00751  (feature_importance 0.0084)
  is_customer_dest     mean|SHAP| 0.00000  (feature_importance 0.0000)

=== TRÊN FRAUD THẬT — SHAP trung bình (đẩy lên = hỗ trợ nhận diện) ===
  type_code            +0.19520
  hour_day             +0.09260
  dest_freq            +0.01281
  dest_type_count      +0

## MLflow Logging

Helper functions để register model lên MLflow. Được gọi ở **cell cuối cùng** (sau Export JSON) khi đã có `final_model`, `FEATURES_CANDIDATES`, `X2_test`.


## Step 5b — Final Model + MLflow

`FEATURES_CANDIDATES` đã được **Step 2 tự động chọn** bằng LOO ablation. Cell này:

- Fit model cuối trên TOÀN train (random-split, cùng test set bước 2)
- Sweep threshold tối ưu **Fβ(1.75)** trên validation set → áp lên test
- Tính **10 metrics chính thức**: roc_auc, confusion_tn/fp/fn/tp, precision, recall, f1, accuracy, pr_auc
- _(MLflow registration nằm ở cell cuối cùng, sau Export JSON)_


In [23]:
# ================================================================
# MLflow Helper Functions
# ================================================================
%pip install -q mlflow

import os
from datetime import datetime, timezone

import mlflow
import mlflow.sklearn
import numpy as np


MLFLOW_TRACKING_URI = os.environ.get(
    "MLFLOW_TRACKING_URI", "http://45.128.222.24:5000"
)
MLFLOW_EXPERIMENT = "paysim-fraud-decision_tree-colab"
REGISTERED_MODEL_NAME = "paysim-fraud-decision_tree"

os.environ.setdefault("MLFLOW_HTTP_REQUEST_TIMEOUT", "30")
os.environ.setdefault("MLFLOW_HTTP_REQUEST_MAX_RETRIES", "2")


def require_registration_inputs(namespace):
    """Fail sớm nếu thiếu biến bắt buộc để register."""
    required = ("final_model", "FEATURES_CANDIDATES", "X2_test")
    missing = [name for name in required if name not in namespace]
    if missing:
        raise NameError(
            "Thiếu biến bắt buộc: "
            + ", ".join(missing)
            + ". Hãy chạy các cell train model trước."
        )


def get_feature_names(model, feature_names):
    return list(
        getattr(model, "feature_names_in_", getattr(model, "feature_name_", feature_names))
    )


def optional_evaluation_metrics(y_true=None, y_pred=None, y_score=None):
    """Tính đúng 10 metrics chính thức (không có fbeta)."""
    if y_true is None or y_pred is None or y_score is None:
        print("[mlflow][optional] Bỏ qua metrics: thiếu y_true/y_pred/y_score")
        return {}

    from sklearn.metrics import (
        accuracy_score, average_precision_score, confusion_matrix,
        f1_score, precision_score, recall_score, roc_auc_score,
    )

    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    if not (len(y_true) == len(y_pred) == len(y_score)):
        print("[mlflow][optional] Bỏ qua metrics: độ dài không khớp")
        return {}

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics = {
        "roc_auc": float(roc_auc_score(y_true, y_score)),
        "confusion_tn": int(tn), "confusion_fp": int(fp),
        "confusion_fn": int(fn), "confusion_tp": int(tp),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "pr_auc": float(average_precision_score(y_true, y_score)),
    }
    return metrics


def optional_log_feature_importance(model, feature_names):
    importance = getattr(model, "feature_importances_", None)
    if importance is None or len(importance) != len(feature_names):
        print("[mlflow][optional] Bỏ qua feature importance")
        return
    mlflow.log_dict(
        {"feature_importance": dict(
            sorted(zip(feature_names, map(float, importance)),
                   key=lambda item: item[1], reverse=True)
        )},
        "evaluation/feature_importance.json",
    )


def register_sklearn_model(model, feature_names, input_frame, *,
                           y_true=None, y_pred=None, y_score=None, threshold=None):
    feature_names = list(feature_names)
    model_input = input_frame.loc[:, feature_names]
    if model_input.empty:
        raise ValueError("X_test[FEATURES] đang rỗng")

    expected_features = get_feature_names(model, feature_names)
    if feature_names != expected_features:
        raise ValueError(f"Thứ tự FEATURES không trùng: {feature_names} != {expected_features}")

    input_example = model_input.iloc[:5].copy().reset_index(drop=True)
    signature = mlflow.models.infer_signature(input_example, model.predict(input_example))

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    run_name = "final_dt__" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({"stage": "final_test", "model_family": "sklearn", "notebook": "decision_tree"})
        params = {"n_features": len(feature_names), "n_rows_evaluated": len(input_frame)}
        if threshold is not None:
            params["threshold"] = float(threshold)
        mlflow.log_params(params)

        metrics = optional_evaluation_metrics(y_true, y_pred, y_score)
        if metrics:
            mlflow.log_metrics(metrics)
        try:
            optional_log_feature_importance(model, feature_names)
        except Exception as exc:
            print(f"[mlflow][optional] Không log được feature importance: {exc}")

        model_info = mlflow.sklearn.log_model(
            model, artifact_path="model",
            registered_model_name=REGISTERED_MODEL_NAME,
            signature=signature, input_example=input_example,

        )
        result = {
            "run_id": run.info.run_id, "experiment_id": run.info.experiment_id,
            "registered_model": REGISTERED_MODEL_NAME, "model_uri": model_info.model_uri,
            **metrics,
        }

    print(f"\n[mlflow] ✓ Registered: {REGISTERED_MODEL_NAME}")
    print(f"[mlflow] run_id: {result['run_id']} | model_uri: {result['model_uri']}")
    return result


Note: you may need to restart the kernel to use updated packages.


In [24]:
# === BƯỚC 5b — Final Model + MLflow ===
# FEATURES_CANDIDATES đã được Step 2 tự động chọn bằng LOO ablation.
# Baseline: fit model trên TOÀN train, sweep threshold Fβ(1.75) trên val,
# áp threshold tối ưu lên test → tính 10 metrics.

import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    f1_score, fbeta_score, precision_recall_curve,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split

print(f"FEATURES_CANDIDATES chính thức ({len(FEATURES_CANDIDATES)} features từ Step 2): {FEATURES_CANDIDATES}")
print(f"BETA = {BETA} (optimization target cho threshold)")

# --- Split train thành train_inner + val để sweep threshold ---
X_inner, X_val, y_inner, y_val = train_test_split(
    X2_train[FEATURES_CANDIDATES], y2_train,
    test_size=0.2, random_state=RANDOM_STATE, stratify=y2_train
)

# Fit model trên train_inner
model_baseline = DecisionTreeClassifier(**DT_PARAMS)
model_baseline.fit(X_inner, y_inner)

# Sweep threshold Fβ(1.75) trên val
p_val_base = model_baseline.predict_proba(X_val)[:, 1]
precisions_v, recalls_v, thresholds_v = precision_recall_curve(y_val, p_val_base)
fb_scores_v = [fbeta_score(y_val, (p_val_base >= t).astype(int), beta=BETA, zero_division=0) for t in thresholds_v]
best_idx_v = int(np.argmax(fb_scores_v))
BASELINE_THRESHOLD = float(thresholds_v[best_idx_v])
print(f"★ Baseline threshold (Fβ({BETA}) trên val): {BASELINE_THRESHOLD:.6f}")

# Refit trên TOÀN train, áp threshold lên test
final_model = DecisionTreeClassifier(**DT_PARAMS)
final_model.fit(X2_train[FEATURES_CANDIDATES], y2_train)
p = final_model.predict_proba(X2_test[FEATURES_CANDIDATES])[:, 1]
pred_baseline = (p >= BASELINE_THRESHOLD).astype(int)

# PR-AUC + R@1% + P@1% (dùng raw score, không phụ thuộc threshold)
pr = average_precision_score(y2_test, p)
n_top1 = max(1, int(np.ceil(len(y2_test) * 0.01)))
top = np.argsort(p)[::-1][:n_top1]
recall1 = y2_test[top].sum() / y2_test.sum()
precision1 = y2_test[top].mean()

# 10 metrics chính thức (với threshold baseline)
tn, fp, fn, tp = confusion_matrix(y2_test, pred_baseline, labels=[0, 1]).ravel()
TEST_METRICS_BASELINE = {
    "roc_auc": float(roc_auc_score(y2_test, p)),
    "confusion_tn": int(tn), "confusion_fp": int(fp),
    "confusion_fn": int(fn), "confusion_tp": int(tp),
    "precision": float(precision_score(y2_test, pred_baseline, zero_division=0)),
    "recall": float(recall_score(y2_test, pred_baseline, zero_division=0)),
    "f1": float(f1_score(y2_test, pred_baseline, zero_division=0)),
    "accuracy": float(accuracy_score(y2_test, pred_baseline)),
    "pr_auc": float(pr),
}

print(f"\n=== 10 METRICS BASELINE (threshold={BASELINE_THRESHOLD:.6f}) ===")
for k, v in TEST_METRICS_BASELINE.items():
    print(f"  {k:20s}: {v:.4f}" if isinstance(v, float) else f"  {k:20s}: {v:,}")
print(f"  PR-AUC: {pr:.4f} | R@1%: {recall1*100:.2f}% | P@1%: {precision1*100:.2f}%")
# MLflow registration đã chuyển xuống sau Export JSON (theo pattern study nb12)
print()
print("✓ Baseline evaluation complete. MLflow registration ở cell cuối.")


FEATURES_CANDIDATES chính thức (6 features từ Step 2): ['hour_day', 'type_code', 'is_customer_dest', 'dest_freq', 'dest_amount_mean', 'dest_type_count']
BETA = 1.75 (optimization target cho threshold)
★ Baseline threshold (Fβ(1.75) trên val): 0.988243

=== 10 METRICS BASELINE (threshold=0.988243) ===
  roc_auc             : 0.9275
  confusion_tn        : 1,268,752
  confusion_fp        : 2,129
  confusion_fn        : 1,104
  confusion_tp        : 539
  precision           : 0.2020
  recall              : 0.3281
  f1                  : 0.2501
  accuracy            : 0.9975
  pr_auc              : 0.1632
  PR-AUC: 0.1632 | R@1%: 55.02% | P@1%: 7.10%

✓ Baseline evaluation complete. MLflow registration ở cell cuối.


In [25]:
# === Export kết quả pipeline chính thức ra JSON ===
import json

results = {
    "pipeline": f"DCAI official: Step2 ({len(FEATURES_CANDIDATES)} features) → Step3 (walk-forward CV) → SHAP → Step 5b (threshold tuning Fβ({BETA})) → MLflow",
    "model": "DecisionTreeClassifier",
    "params": DT_PARAMS,
    "features_final": FEATURES_CANDIDATES,
    "n_features": len(FEATURES_CANDIDATES),
    "beta": BETA,
    "baseline_threshold": round(BASELINE_THRESHOLD, 6),
    "test_metrics_baseline": TEST_METRICS_BASELINE,
}

out_path = "decision_tree_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"✓ Saved to {out_path}")

# ================================================================
# MLflow Registration (bước CUỐI CÙNG — sau Export JSON)
# ================================================================
require_registration_inputs(globals())

MLFLOW_RESULT = register_sklearn_model(
    model=final_model,
    feature_names=FEATURES_CANDIDATES,
    input_frame=X2_test,
    y_true=y2_test,
    y_pred=pred_baseline,
    y_score=p,
    threshold=BASELINE_THRESHOLD,
)

# Cập nhật JSON với MLflow info sau khi register thành công
results["mlflow"] = {
    "run_id": MLFLOW_RESULT["run_id"],
    "registered_model": MLFLOW_RESULT["registered_model"],
    "model_uri": MLFLOW_RESULT["model_uri"],
}
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print()
print(f"★ MLflow registered. Run ID: {MLFLOW_RESULT['run_id']}")
print(f"✓ JSON updated with MLflow info")
print(json.dumps(results, indent=2, ensure_ascii=False))

MLFLOW_RESULT


✓ Saved to decision_tree_results.json


c:\Users\FPT\miniconda3\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/09/03 15:35:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Registered model 'paysim-fraud-decision_tree' already exists. Creating a new version of this model...
2026/09/03 15:35:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: paysim-fraud-decision_tree, version 6
Created version '6' of model 'paysim-fraud-decision_tree'.


🏃 View run final_dt__20260903-083520 at: http://45.128.222.24:5000/#/experiments/6/runs/c13a3b3f8bbd447485c80c27c98e3a54
🧪 View experiment at: http://45.128.222.24:5000/#/experiments/6

[mlflow] ✓ Registered: paysim-fraud-decision_tree
[mlflow] run_id: c13a3b3f8bbd447485c80c27c98e3a54 | model_uri: models:/m-18db6bfccb0a4bd790de4e444c6e699b

★ MLflow registered. Run ID: c13a3b3f8bbd447485c80c27c98e3a54
✓ JSON updated with MLflow info
{
  "pipeline": "DCAI official: Step2 (6 features) → Step3 (walk-forward CV) → SHAP → Step 5b (threshold tuning Fβ(1.75)) → MLflow",
  "model": "DecisionTreeClassifier",
  "params": {
    "max_depth": 6,
    "min_samples_leaf": 50,
    "min_samples_split": 100,
    "class_weight": "balanced",
    "random_state": 42
  },
  "features_final": [
    "hour_day",
    "type_code",
    "is_customer_dest",
    "dest_freq",
    "dest_amount_mean",
    "dest_type_count"
  ],
  "n_features": 6,
  "beta": 1.75,
  "baseline_threshold": 0.988243,
  "test_metrics_baseline"

{'run_id': 'c13a3b3f8bbd447485c80c27c98e3a54',
 'experiment_id': '6',
 'registered_model': 'paysim-fraud-decision_tree',
 'model_uri': 'models:/m-18db6bfccb0a4bd790de4e444c6e699b',
 'roc_auc': 0.927549657166215,
 'confusion_tn': 1268752,
 'confusion_fp': 2129,
 'confusion_fn': 1104,
 'confusion_tp': 539,
 'precision': 0.202023988005997,
 'recall': 0.3280584297017651,
 'f1': 0.25005799118533983,
 'accuracy': 0.9974593799409677,
 'pr_auc': 0.16319987606675251}